# STEP8. 정책국면별 산업 금리민감도

## 분석 질문
- 같은 금리변화라도 통화정책 국면(완화·인하 vs 인상·긴축)에 따라 산업별 금리베타가 달라지는가.
- 관련 가설: H2 (국면 의존성, 모형 D)

## 입력

| 구분 | 경로 |
|---|---|
| 산업 금리 패널 | `config.PROCESSED_DIR / "industry_rate_panel_daily.csv"` |
| 정책국면 정의 | `config.RAW_DIR / "market" / "model_D_policy_regime_daily.csv"` (없으면 합의 날짜로 자동 생성) |

## 출력

| 파일 | 위치 |
|---|---|
| `regime_beta_long.csv` · `regime_beta_wide.csv` | ``config.H2_TABDIR`` |
| `regime_policy_direction_wald.csv` | ``config.H2_TABDIR`` |
| `regime_equality_wald.csv` | ``config.H2_TABDIR`` |
| `regime_final_summary.csv` | ``config.H2_TABDIR`` |
| `regime_{산업}_beta_ci.png` (7개) | ``config.H2_FIGDIR`` |

## 전체 수행 흐름
1. 정책국면 병합·검증
2. D-1: 산업별 통합회귀로 5개 국면 금리베타 β₁~β₅ 추정 (국면별 절편·시장베타·금리베타)
3. D-1 결과표·95% CI 그래프
4. D-2: 인상·긴축(2·4기) vs 완화·인하(1·3·5기) 평균 금리베타 차이 Wald/F 검정
5. D-3: β₁=β₂=β₃=β₄=β₅ 전체 동일성 Wald/F 검정
6. D-2·D-3 각각 7개 산업 BH(FDR) 보정

핵심 금리변수는 `d_y3`(국고채 3년 금리 일별 변화)이고, `regime`은 한국은행 정책 사이클 기준 국면 변수다.

In [ ]:
# 0. 환경 설정 · 경로 · 라이브러리

# --- 저장소 루트 부트스트랩 (모든 노트북 공통, 수정 금지) ---
# 저장소 루트에서 `pip install -e .` 를 한 번 실행했다면 아래 탐색은 건너뛴다.
import sys
from pathlib import Path

_anchors = []
if "__vsc_ipynb_file__" in globals():          # VS Code Notebook
    _anchors.append(Path(globals()["__vsc_ipynb_file__"]).resolve().parent)
if globals().get("_dh"):                        # IPython 커널 시작 폴더
    _anchors.append(Path(globals()["_dh"][0]).resolve())
_anchors.append(Path.cwd().resolve())           # 최후 수단

# config.py 와 requirements.txt 를 '둘 다' 가진 폴더만 저장소 루트로 인정한다.
_root = next(
    (p for a in _anchors for p in [a, *a.parents]
     if (p / "config.py").is_file() and (p / "requirements.txt").is_file()),
    None,
)
if _root is not None:
    if str(_root) in sys.path:
        sys.path.remove(str(_root))
    sys.path.insert(0, str(_root))

import config

if _root is not None and Path(config.__file__).resolve().parent != _root:
    raise ImportError(
        f"의도하지 않은 config.py 가 import 되었습니다: {config.__file__}\n"
        f"저장소 루트({_root})에서 `pip install -e .` 를 실행하세요."
    )
# --- 부트스트랩 끝 ---

# 한글 폰트 공통 설정 (config.py 5-c절).
config.setup_korean_font()

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")

# 공통 상수는 config에서 가져온다 (STYLE_GUIDE 13·14절).
HAC_LAGS = config.HAC_MAXLAGS
ALPHA = config.BH_ALPHA

# 경로는 전부 config에서 가져온다. 개인 PC 절대경로를 출력에 남기지 않는다(config._rel 사용).
PANEL_FILE = config.PROCESSED_DIR / "industry_rate_panel_daily.csv"
REGIME_FILE = config.RAW_DIR / "market" / "model_D_policy_regime_daily.csv"

# 모형 D 산출물은 H2(비대칭·국면) 폴더에 regime_ 접두로 저장한다 (HYPOTHESIS_MODEL_MAP).
OUTPUT_DIR = config.H2_TABDIR
FIG_DIR = config.H2_FIGDIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("config      :", config._rel(Path(config.__file__)))
print("PANEL_FILE  :", config._rel(PANEL_FILE), "(존재:", PANEL_FILE.exists(), ")")
print("REGIME_FILE :", config._rel(REGIME_FILE), "(존재:", REGIME_FILE.exists(), ")")
print("OUTPUT_DIR  :", config._rel(OUTPUT_DIR))

## 1. 기존 최종 패널 불러오기

`src/data_prep.py`의 최종 산출물인 `industry_rate_panel_daily.csv`에는 분석에 필요한 핵심 변수인
`date`, `industry`, `industry_return`, `kospi_return`, `d_y3`가 이미 생성되어 있다.

In [3]:
# 1-1. 최종 패널 불러오기
if not PANEL_FILE.exists():
    raise FileNotFoundError(
        f"최종 패널을 찾을 수 없습니다: {PANEL_FILE}\n"
        "먼저 src/data_prep.py의 run_data_preparation() / run_variable_generation()을 실행하세요."
    )

panel = pd.read_csv(PANEL_FILE, encoding="utf-8-sig")
panel["date"] = pd.to_datetime(panel["date"])

required_cols = ["date", "industry", "industry_return", "kospi_return", "d_y3"]
missing_cols = [c for c in required_cols if c not in panel.columns]
if missing_cols:
    raise ValueError(f"최종 패널에 필요한 컬럼이 없습니다: {missing_cols}")

print("패널 크기:", panel.shape)
print("기간:", panel["date"].min().date(), "~", panel["date"].max().date())
print("산업 수:", panel["industry"].nunique())
print("산업:", sorted(panel["industry"].dropna().unique().tolist()))
display(panel[required_cols].head())

패널 크기: (17164, 12)
기간: 2016-01-04 ~ 2025-12-30
산업 수: 7
산업: ['건설업', '금융업', '운수장비', '의약품', '전기전자', '철강금속', '화학']


,date,industry,industry_return,kospi_return,d_y3
0,2016-01-04,건설업,NaN,NaN,NaN
1,2016-01-04,금융업,NaN,NaN,NaN
2,2016-01-04,운수장비,NaN,NaN,NaN
3,2016-01-04,의약품,NaN,NaN,NaN
4,2016-01-04,전기전자,NaN,NaN,NaN


## 2. 정책국면 데이터 준비 및 병합

정책국면 정의:

- **1기:** 2016-01-04 ~ 2017-11-29 → 완화·인하 (`tightening=0`)
- **2기:** 2017-11-30 ~ 2019-07-17 → 인상·긴축 (`tightening=1`)
- **3기:** 2019-07-18 ~ 2021-08-25 → 완화·인하 (`tightening=0`)
- **4기:** 2021-08-26 ~ 2024-10-10 → 인상·긴축 (`tightening=1`)
- **5기:** 2024-10-11 ~ 2025-12-30 → 완화·인하 (`tightening=0`)

In [4]:
# 2-1. 정책국면 데이터 불러오기 또는 자동 생성
def find_regime_file():
    if REGIME_FILE is not None:
        p = Path(REGIME_FILE)
        return p if p.exists() else None

    candidates = [
        config.PROCESSED_DIR / "model_D_policy_regime_daily.csv",
        config.PROCESSED_DIR / "policy_regime_daily.csv",
        config.INTERIM_DIR / "model_D_policy_regime_daily.csv",
        config.INTERIM_DIR / "policy_regime_daily.csv",
        Path.cwd() / "model_D_policy_regime_daily.csv",
        Path.cwd() / "policy_regime_daily.csv",
    ]
    for p in candidates:
        if p.exists():
            return p
    return None


def build_regime_map_from_dates():
    specs = [
        (1, "2016-01-04", "2017-11-29", 0, "저금리·완화 국면", "완화·인하"),
        (2, "2017-11-30", "2019-07-17", 1, "인상·정상화 사이클", "인상·긴축"),
        (3, "2019-07-18", "2021-08-25", 0, "인하·초저금리 사이클", "완화·인하"),
        (4, "2021-08-26", "2024-10-10", 1, "인상·고금리 유지 사이클", "인상·긴축"),
        (5, "2024-10-11", "2025-12-30", 0, "인하 재개 사이클", "완화·인하"),
    ]

    parts = []
    for regime, start, end, tightening, policy_cycle, direction in specs:
        dates = pd.date_range(start=start, end=end, freq="D")
        parts.append(pd.DataFrame({
            "date": dates,
            "regime": regime,
            "tightening": tightening,
            "policy_cycle": policy_cycle,
            "direction": direction,
        }))
    return pd.concat(parts, ignore_index=True)


regime_path = find_regime_file()

if regime_path is not None:
    regime_map = pd.read_csv(regime_path, encoding="utf-8-sig")
    regime_map["date"] = pd.to_datetime(regime_map["date"])
    print("정책국면 파일 사용:", config._rel(Path(regime_path)))
else:
    regime_map = build_regime_map_from_dates()
    print("정책국면 파일을 찾지 못해 합의된 날짜 기준으로 자동 생성했습니다.")

keep_cols = [c for c in ["date", "regime", "tightening", "policy_cycle", "direction"] if c in regime_map.columns]
regime_map = regime_map[keep_cols].copy()

for c in ["date", "regime", "tightening"]:
    if c not in regime_map.columns:
        raise ValueError(f"정책국면 데이터에 필수 컬럼 `{c}`가 없습니다.")

if regime_map["date"].duplicated().any():
    dup = regime_map.loc[regime_map["date"].duplicated(), "date"].head().tolist()
    raise ValueError(f"정책국면 데이터에 중복 날짜가 있습니다. 예: {dup}")

display(regime_map.head())

정책국면 파일 사용: <PROJECT_ROOT>\data\raw\market\model_D_policy_regime_daily.csv


,date,regime,tightening,policy_cycle,direction
0,2016-01-04,1,0,저금리·완화 국면,완화·인하
1,2016-01-05,1,0,저금리·완화 국면,완화·인하
2,2016-01-06,1,0,저금리·완화 국면,완화·인하
3,2016-01-07,1,0,저금리·완화 국면,완화·인하
4,2016-01-08,1,0,저금리·완화 국면,완화·인하


In [5]:
# 2-2. 기존 패널에 regime / tightening 병합
model_d_panel = panel.merge(
    regime_map,
    on="date",
    how="left",
    validate="many_to_one",
)

analysis_mask = model_d_panel["date"].between("2016-01-04", "2025-12-30")
missing_regime = model_d_panel.loc[analysis_mask, "regime"].isna().sum()

print("병합 후 크기:", model_d_panel.shape)
print("분석기간 내 regime 누락 행:", missing_regime)

if missing_regime > 0:
    bad_dates = (
        model_d_panel.loc[analysis_mask & model_d_panel["regime"].isna(), "date"]
        .drop_duplicates()
        .sort_values()
        .head(20)
        .tolist()
    )
    raise ValueError(f"정책국면이 매핑되지 않은 날짜가 있습니다. 예: {bad_dates}")

model_d_panel["regime"] = model_d_panel["regime"].astype(int)
model_d_panel["tightening"] = model_d_panel["tightening"].astype(int)

display(model_d_panel[[
    "date", "industry", "industry_return", "kospi_return", "d_y3",
    "regime", "tightening"
]].head())

병합 후 크기: (17164, 16)
분석기간 내 regime 누락 행: 0


,date,industry,industry_return,kospi_return,d_y3,regime,tightening
0,2016-01-04,건설업,NaN,NaN,NaN,1,0
1,2016-01-04,금융업,NaN,NaN,NaN,1,0
2,2016-01-04,운수장비,NaN,NaN,NaN,1,0
3,2016-01-04,의약품,NaN,NaN,NaN,1,0
4,2016-01-04,전기전자,NaN,NaN,NaN,1,0


### 2-3. 가장 먼저 확인할 것 — 국면별 날짜 수와 산업별 관측치

회귀 전에 각 국면이 정상적으로 배정되었는지 확인한다.

In [6]:
# 2-3. 국면별 날짜 수
regime_date_counts = (
    model_d_panel[["date", "regime", "tightening"]]
    .drop_duplicates("date")
    .groupby(["regime", "tightening"])
    .size()
    .rename("n_dates")
    .reset_index()
)

print("[국면별 실제 패널 날짜 수]")
display(regime_date_counts)

industry_regime_counts = (
    model_d_panel
    .groupby(["industry", "regime"])
    .size()
    .unstack("regime")
    .rename(columns=lambda x: f"regime_{x}")
)

print("[산업별 국면 관측치 수]")
display(industry_regime_counts)

complete_counts = (
    model_d_panel
    .dropna(subset=["industry_return", "kospi_return", "d_y3", "regime"])
    .groupby(["industry", "regime"])
    .size()
    .unstack("regime")
    .rename(columns=lambda x: f"regime_{x}")
)

print("[산업별 국면 회귀 유효 관측치 수]")
display(complete_counts)

[국면별 실제 패널 날짜 수]


,regime,tightening,n_dates
0,1,0,469
1,2,1,398
2,3,0,522
3,4,1,765
4,5,0,298


[산업별 국면 관측치 수]


regime,regime_1,regime_2,regime_3,regime_4,regime_5
industry,,,,,
건설업,469,398,522,765,298
금융업,469,398,522,765,298
운수장비,469,398,522,765,298
의약품,469,398,522,765,298
전기전자,469,398,522,765,298
철강금속,469,398,522,765,298
화학,469,398,522,765,298


[산업별 국면 회귀 유효 관측치 수]


regime,regime_1,regime_2,regime_3,regime_4,regime_5
industry,,,,,
건설업,468,398,522,765,298
금융업,468,398,522,765,298
운수장비,468,398,522,765,298
의약품,468,398,522,765,298
전기전자,468,398,522,765,298
철강금속,468,398,522,765,298
화학,468,398,522,765,298


# D-1. 5개 국면별 금리베타 통합회귀

각 산업별로 **회귀는 한 번만** 수행한다.

국면별로 다음 세 가지를 모두 다르게 허용한다.

- 절편
- KOSPI 시장베타
- 국고채 3년 금리베타

각 산업의 회귀식은 다음과 같다.

$$
R_{i,t}
=
\sum_{r=1}^{5}
\left[
\alpha_{i,r} D_{r,t}
+
\beta^{M}_{i,r}
\left(D_{r,t} R_{M,t}\right)
+
\beta^{Y}_{i,r}
\left(D_{r,t} \Delta y_{3,t}\right)
\right]
+
\epsilon_{i,t}
$$

여기서,

- $R_{i,t}$ : 산업 $i$의 일별 수익률
- $R_{M,t}$ : KOSPI 일별 수익률
- $\Delta y_{3,t}$ : 국고채 3년 금리의 일별 변화
- $D_{r,t}$ : 해당 날짜가 정책국면 $r$에 속하면 1, 아니면 0
- $\alpha_{i,r}$ : 산업 $i$의 국면 $r$별 절편
- $\beta^{M}_{i,r}$ : 산업 $i$의 국면 $r$별 시장베타
- $\beta^{Y}_{i,r}$ : 산업 $i$의 국면 $r$별 금리베타

**관심 대상은 `rate_r1` ~ `rate_r5`의 계수인  
$\beta^{Y}_{i,1}$ ~ $\beta^{Y}_{i,5}$이다.**

In [7]:
# D-1-1. 통합회귀 함수
REGIMES = [1, 2, 3, 4, 5]

def build_regime_design(df):
    # 전체 절편은 넣지 않고 regime별 절편 5개를 직접 넣는다.
    X = pd.DataFrame(index=df.index)

    for r in REGIMES:
        d = (df["regime"] == r).astype(float)
        X[f"const_r{r}"] = d
        X[f"mkt_r{r}"] = d * df["kospi_return"]
        X[f"rate_r{r}"] = d * df["d_y3"]

    return X


def run_regime_model(data, industry, hac_lags=HAC_LAGS):
    sub = (
        data.loc[
            data["industry"] == industry,
            ["date", "industry_return", "kospi_return", "d_y3", "regime"]
        ]
        .dropna()
        .sort_values("date")
        .copy()
    )

    present_regimes = sorted(sub["regime"].unique().tolist())
    if present_regimes != REGIMES:
        raise ValueError(
            f"{industry}: 5개 국면이 모두 존재하지 않습니다. 현재 국면={present_regimes}"
        )

    X = build_regime_design(sub)
    y = sub["industry_return"]

    model = sm.OLS(y, X)
    result = model.fit(
        cov_type="HAC",
        cov_kwds={"maxlags": hac_lags},
    )
    return result, sub


preferred_order = ["건설업", "금융업", "화학", "운수장비", "전기전자", "철강금속", "의약품"]
industries = [x for x in preferred_order if x in model_d_panel["industry"].unique()]
industries += [
    x for x in sorted(model_d_panel["industry"].unique())
    if x not in industries
]

model_results = {}
model_samples = {}

for industry in industries:
    res, sample = run_regime_model(model_d_panel, industry)
    model_results[industry] = res
    model_samples[industry] = sample
    print(f"{industry}: n={int(res.nobs)}, adj.R²={res.rsquared_adj:.4f}")

print("\n완료된 산업 수:", len(model_results))

건설업: n=2451, adj.R²=0.4042
금융업: n=2451, adj.R²=0.6259
화학: n=2451, adj.R²=0.6452
운수장비: n=2451, adj.R²=0.4886
전기전자: n=2451, adj.R²=0.7275
철강금속: n=2451, adj.R²=0.3853
의약품: n=2451, adj.R²=0.2499

완료된 산업 수: 7


## D-1 결과표 — β₁~β₅, HAC 표준오차, p값, 95% CI

In [8]:
# D-1-2. 국면별 금리베타 추출
def result_series(result):
    names = result.model.exog_names
    params = pd.Series(np.asarray(result.params), index=names)
    bse = pd.Series(np.asarray(result.bse), index=names)
    pvalues = pd.Series(np.asarray(result.pvalues), index=names)
    ci = pd.DataFrame(
        np.asarray(result.conf_int(alpha=ALPHA)),
        index=names,
        columns=["ci_low", "ci_high"],
    )
    return params, bse, pvalues, ci


d1_rows = []

for industry, res in model_results.items():
    params, bse, pvalues, ci = result_series(res)

    for r in REGIMES:
        name = f"rate_r{r}"
        d1_rows.append({
            "industry": industry,
            "regime": r,
            "beta": params[name],
            "hac_se": bse[name],
            "p_value": pvalues[name],
            "ci_low": ci.loc[name, "ci_low"],
            "ci_high": ci.loc[name, "ci_high"],
            "nobs_total_industry": int(res.nobs),
            "nobs_regime": int((model_samples[industry]["regime"] == r).sum()),
        })

d1_long = pd.DataFrame(d1_rows)

d1_beta_wide = (
    d1_long
    .pivot(index="industry", columns="regime", values="beta")
    .rename(columns=lambda r: f"{r}기 β")
    .reindex(industries)
)

print("[D-1 국면별 금리베타]")
display(d1_beta_wide.round(6))

print("[D-1 상세 결과]")
display(d1_long.round(6))

d1_long.to_csv(OUTPUT_DIR / "regime_beta_long.csv", index=False, encoding="utf-8-sig")
d1_beta_wide.to_csv(OUTPUT_DIR / "regime_beta_wide.csv", encoding="utf-8-sig")

[D-1 국면별 금리베타]


regime,1기 β,2기 β,3기 β,4기 β,5기 β
industry,,,,,
건설업,0.036986,0.095305,-0.010244,-0.002941,-0.039238
금융업,0.050240,0.024372,0.001273,0.006019,0.034307
화학,0.005831,0.017157,-0.017168,0.007710,-0.019556
운수장비,0.081579,0.034323,-0.017115,0.016247,0.002448
전기전자,-0.054118,-0.042118,0.022710,0.002758,-0.000860
철강금속,0.070965,0.042810,0.009131,0.024990,-0.057606
의약품,-0.023950,-0.032675,0.001379,-0.035623,-0.029533


[D-1 상세 결과]


,industry,regime,beta,hac_se,p_value,ci_low,ci_high,nobs_total_industry,nobs_regime
0,건설업,1,0.036986,0.022547,0.100912,-0.007204,0.081177,2451,468
1,건설업,2,0.095305,0.038858,0.014181,0.019145,0.171466,2451,398
2,건설업,3,-0.010244,0.027542,0.709930,-0.064226,0.043737,2451,522
3,건설업,4,-0.002941,0.009504,0.756993,-0.021567,0.015686,2451,765
4,건설업,5,-0.039238,0.025906,0.129875,-0.090014,0.011538,2451,298
5,금융업,1,0.050240,0.014782,0.000677,0.021267,0.079212,2451,468
6,금융업,2,0.024372,0.015056,0.105487,-0.005136,0.053881,2451,398
7,금융업,3,0.001273,0.014325,0.929184,-0.026803,0.029350,2451,522
8,금융업,4,0.006019,0.003937,0.126315,-0.001698,0.013735,2451,765
9,금융업,5,0.034307,0.021873,0.116775,-0.008563,0.077177,2451,298


## D-1 시각화 — 산업별 5개 β + 95% 신뢰구간

각 산업마다 별도의 그래프로 저장한다.

- X축: 정책국면 1~5
- Y축: 국고채 3년 금리베타
- 점: 추정 β
- 세로선: 95% CI
- 0선: 금리효과가 없는 기준

In [ ]:
# D-1-3. 산업별 β + 95% CI 그래프

# 한글 폰트·음수기호는 config.setup_korean_font()(0셀)에서 이미 설정됨.


# --------------------------------------------------
# 2. 산업별 β + 95% CI 그래프
# --------------------------------------------------
for industry in industries:

    sub = (
        d1_long[d1_long["industry"] == industry]
        .sort_values("regime")
    )

    x = sub["regime"].to_numpy()
    y = sub["beta"].to_numpy()

    lower = y - sub["ci_low"].to_numpy()
    upper = sub["ci_high"].to_numpy() - y

    fig, ax = plt.subplots(figsize=(8, 4.8))

    ax.errorbar(
        x,
        y,
        yerr=np.vstack([lower, upper]),
        fmt="o",
        capsize=5,
        markersize=6,
    )

    # β = 0 기준선
    ax.axhline(0, linewidth=1)

    ax.set_xticks(REGIMES)
    ax.set_xlabel("정책국면")
    ax.set_ylabel("국고채 3년 금리베타")
    ax.set_title(f"{industry} — 정책국면별 금리베타 (95% CI)")

    ax.grid(axis="y", alpha=0.25)

    fig.tight_layout()

    # 저장
    out = FIG_DIR / f"regime_{industry}_beta_ci.png"
    fig.savefig(out, dpi=160, bbox_inches="tight")

    plt.show()
    plt.close(fig)


print("그래프 저장 폴더:", config._rel(FIG_DIR))

# D-2. 정책 방향 검정

D-1에서 추정한 **같은 통합회귀 결과**를 이용하여 선형제약을 검정한다.  
즉, D-2를 위해 새로운 회귀를 다시 추정하지 않는다.

정책 방향은 다음과 같이 구분한다.

- **인상·긴축 국면:** 2기, 4기
- **완화·인하 국면:** 1기, 3기, 5기

### 검정하고자 하는 질문

> 인상·긴축 국면의 평균 금리민감도와  
> 완화·인하 국면의 평균 금리민감도는 서로 다른가?

### 귀무가설

$$
H_0:
\frac{\beta_2+\beta_4}{2}
=
\frac{\beta_1+\beta_3+\beta_5}{3}
$$

왼쪽은 **인상·긴축 국면의 평균 금리베타**이고,

$$
\frac{\beta_2+\beta_4}{2}
$$

오른쪽은 **완화·인하 국면의 평균 금리베타**이다.

$$
\frac{\beta_1+\beta_3+\beta_5}{3}
$$

따라서 D-2에서는 두 정책 방향의 금리민감도가 동일하다는 귀무가설을  
**Wald/F 검정**으로 확인한다.

- 귀무가설을 기각하면 → 인상·긴축과 완화·인하 사이에 금리민감도 차이가 존재
- 귀무가설을 기각하지 못하면 → 정책 방향에 따른 차이를 통계적으로 확인하기 어려움

7개 산업에서 각각 검정한 뒤, 다중검정 문제를 고려하여 **BH(FDR) 보정**을 적용한다.

In [11]:
# D-2-1. 인상·긴축 vs 완화·인하 Wald/F 검정
def make_contrast_vector(result, weights):
    names = result.model.exog_names
    R = np.zeros((1, len(names)))

    for regime, weight in weights.items():
        pname = f"rate_r{regime}"
        R[0, names.index(pname)] = weight

    return R


d2_weights = {
    1: -1/3,
    2: +1/2,
    3: -1/3,
    4: +1/2,
    5: -1/3,
}

d2_rows = []

for industry, res in model_results.items():
    params, _, _, _ = result_series(res)

    beta_tight = (params["rate_r2"] + params["rate_r4"]) / 2
    beta_ease = (params["rate_r1"] + params["rate_r3"] + params["rate_r5"]) / 3
    diff = beta_tight - beta_ease

    R = make_contrast_vector(res, d2_weights)
    test = res.wald_test(R, use_f=True, scalar=True)

    d2_rows.append({
        "industry": industry,
        "beta_tight_mean": beta_tight,
        "beta_ease_mean": beta_ease,
        "difference_tight_minus_ease": diff,
        "F_stat": float(test.statistic),
        "p_value": float(test.pvalue),
    })

d2_results = pd.DataFrame(d2_rows)

reject, bh_p, _, _ = multipletests(
    d2_results["p_value"].to_numpy(),
    alpha=ALPHA,
    method="fdr_bh",
)
d2_results["bh_p"] = bh_p
d2_results["bh_sig_5pct"] = reject

d2_results = d2_results.sort_values("bh_p").reset_index(drop=True)

print("[D-2 정책 방향 검정]")
display(d2_results.round(6))

d2_results.to_csv(
    OUTPUT_DIR / "regime_policy_direction_wald.csv",
    index=False,
    encoding="utf-8-sig",
)

[D-2 정책 방향 검정]


,industry,beta_tight_mean,beta_ease_mean,difference_tight_minus_ease,F_stat,p_value,bh_p,bh_sig_5pct
0,건설업,0.046182,-0.004165,0.050348,4.130536,0.042224,0.295570,False
1,화학,0.012433,-0.010297,0.022731,2.164126,0.141394,0.494879,False
2,금융업,0.015196,0.028607,-0.013411,1.121985,0.289596,0.517138,False
3,철강금속,0.033900,0.007497,0.026403,1.094830,0.295507,0.517138,False
4,의약품,-0.034149,-0.017368,-0.016781,0.232326,0.629848,0.734823,False
5,전기전자,-0.019680,-0.010756,-0.008924,0.315420,0.574425,0.734823,False
6,운수장비,0.025285,0.022304,0.002981,0.025334,0.873551,0.873551,False


### D-2 해석 원칙

- BH 보정 후 유의:
  - 해당 산업의 국고채 3년 금리민감도가 **인상·긴축 vs 완화·인하 정책 방향에 따라 체계적으로 다르다**.
- 비유의:
  - 정책 방향만으로 금리민감도 차이를 설명할 통계적 근거가 부족하다.

`difference_tight_minus_ease`의 부호를 함께 확인하면 어느 방향에서 금리베타가 더 큰지도 볼 수 있다.

# D-3. 5개 정책국면 전체 동일성 검정

D-1의 같은 회귀 결과에서 다음 귀무가설을 공동 검정한다.

\[
H_0:\beta_1=\beta_2=\beta_3=\beta_4=\beta_5
\]

이를 다음 4개 제약으로 구현한다.

- β₁ − β₂ = 0
- β₁ − β₃ = 0
- β₁ − β₄ = 0
- β₁ − β₅ = 0

유의하면 **적어도 하나의 정책사이클에서 금리베타가 다르다**는 뜻이다.

In [12]:
# D-3-1. β1=β2=β3=β4=β5 공동 Wald/F 검정
def make_equal_beta_restrictions(result):
    names = result.model.exog_names
    R = np.zeros((4, len(names)))

    comparisons = [2, 3, 4, 5]
    for row_idx, r in enumerate(comparisons):
        R[row_idx, names.index("rate_r1")] = +1
        R[row_idx, names.index(f"rate_r{r}")] = -1

    return R


d3_rows = []

for industry, res in model_results.items():
    R = make_equal_beta_restrictions(res)
    test = res.wald_test(R, use_f=True, scalar=True)

    d3_rows.append({
        "industry": industry,
        "F_stat": float(test.statistic),
        "df_num": 4,
        "df_denom": float(getattr(test, "df_denom", np.nan)),
        "p_value": float(test.pvalue),
    })

d3_results = pd.DataFrame(d3_rows)

reject, bh_p, _, _ = multipletests(
    d3_results["p_value"].to_numpy(),
    alpha=ALPHA,
    method="fdr_bh",
)
d3_results["bh_p"] = bh_p
d3_results["bh_sig_5pct"] = reject

d3_results = d3_results.sort_values("bh_p").reset_index(drop=True)

print("[D-3 5개 국면 전체 동일성 검정]")
display(d3_results.round(6))

d3_results.to_csv(
    OUTPUT_DIR / "regime_equality_wald.csv",
    index=False,
    encoding="utf-8-sig",
)

[D-3 5개 국면 전체 동일성 검정]


,industry,F_stat,df_num,df_denom,p_value,bh_p,bh_sig_5pct
0,건설업,2.783899,4,2436.0,0.025304,0.065002,False
1,금융업,2.726841,4,2436.0,0.027858,0.065002,False
2,전기전자,2.942846,4,2436.0,0.019328,0.065002,False
3,운수장비,2.216737,4,2436.0,0.064841,0.113473,False
4,철강금속,1.783951,4,2436.0,0.129262,0.180966,False
5,화학,1.055178,4,2436.0,0.377205,0.440072,False
6,의약품,0.372918,4,2436.0,0.828088,0.828088,False


### D-3 해석 원칙

- BH 보정 후 유의:
  - 해당 산업의 금리민감도는 2016~2025년 동안 하나의 고정된 β로 보기 어렵다.
  - **적어도 하나의 정책사이클에서 금리베타가 다르다.**
  - 단, 이 검정만으로 “인상기에 더 민감하다”라고 해석하면 안 된다. 정책 방향 차이는 D-2에서 판단한다.

- 비유의:
  - 5개 정책사이클 사이의 금리베타 차이를 통계적으로 확인하지 못했다.

# 최종 요약표

D-1의 5개 β와 D-2·D-3의 검정 결과를 한 표에 합친다.

In [ ]:
# 최종 요약표
summary = d1_beta_wide.reset_index()

summary = summary.merge(
    d2_results[[
        "industry",
        "beta_tight_mean",
        "beta_ease_mean",
        "difference_tight_minus_ease",
        "p_value",
        "bh_p",
        "bh_sig_5pct",
    ]].rename(columns={
        "p_value": "D2_p",
        "bh_p": "D2_bh_p",
        "bh_sig_5pct": "D2_sig",
    }),
    on="industry",
    how="left",
)

summary = summary.merge(
    d3_results[[
        "industry",
        "F_stat",
        "p_value",
        "bh_p",
        "bh_sig_5pct",
    ]].rename(columns={
        "F_stat": "D3_F",
        "p_value": "D3_p",
        "bh_p": "D3_bh_p",
        "bh_sig_5pct": "D3_sig",
    }),
    on="industry",
    how="left",
)

def classify_row(row):
    d2 = bool(row["D2_sig"])
    d3 = bool(row["D3_sig"])

    if d2 and d3:
        return "정책 방향 차이 + 개별 사이클 차이"
    if d2 and not d3:
        return "정책 방향의 체계적 차이"
    if (not d2) and d3:
        return "사이클 의존성 존재, 단순 인상/인하로는 설명 어려움"
    return "국면 의존성의 통계적 증거 약함"

summary["interpretation"] = summary.apply(classify_row, axis=1)

print("[모형 D 최종 요약]")
display(summary.round(6))

summary.to_csv(
    OUTPUT_DIR / "regime_final_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

print("모든 결과 저장 완료:", config._rel(OUTPUT_DIR))

## 결과를 읽는 순서

1. **D-1**에서 각 산업의 β₁~β₅ 부호·크기·95% CI를 확인한다.
2. **D-2**에서 인상·긴축과 완화·인하 사이에 체계적인 차이가 있는지 본다.
3. **D-3**에서 정책 방향을 떠나 개별 정책사이클 자체에 따라 β가 달라지는지 본다.
4. D-2와 D-3의 **BH 보정 p값**을 최종 판단 기준으로 사용한다.

이 노트북은 모형 B의 결과를 전제로 하지 않고, `d_y3`를 이용한 독립적인 정책국면 분석으로 설계되어 있다.